Programmer: python_scripts (Abhijith Warrier)

***🔬 Python Code Lab — Does `__slots__` Really Save Memory?***

Python's `__slots__` is often used to reduce the memory overhead of class instances.

Instead of simply assuming it saves memory, this experiment compares regular Python classes with slotted classes and measures the difference as the number of objects increases.

----

## ❓ Question

Does using `__slots__` actually reduce the memory consumed by Python objects?

We'll create equivalent regular and slotted classes, generate thousands of instances, and measure their memory usage under the same conditions.

---

## 🧪 Experiment Setup

We'll create two classes containing exactly the same attributes:

- `RegularUser` — a normal Python class
- `SlottedUser` — the same class using `__slots__`

We'll use `tracemalloc` to measure memory allocated while creating collections of these objects.

---

In [1]:
import gc
import tracemalloc

---

## ▶️ Experiment 1 — Creating Equivalent Classes

Both classes store the same three attributes.

The only structural difference is that `SlottedUser` declares those attributes using `__slots__`.

In [2]:
class RegularUser:
    def __init__(self, user_id, name, email):
        self.user_id = user_id
        self.name = name
        self.email = email


class SlottedUser:
    __slots__ = ("user_id", "name", "email")

    def __init__(self, user_id, name, email):
        self.user_id = user_id
        self.name = name
        self.email = email

---

## ▶️ Experiment 2 — Measuring Memory Usage

The function below starts a fresh `tracemalloc` session, creates the requested number of objects, and records the traced memory while those objects are still alive.

Garbage collection is triggered before every measurement to reduce interference from objects left behind by previous runs.

In [3]:
def measure_memory(user_class, count):
    gc.collect()
    tracemalloc.start()

    users = [
        user_class(i, f"User {i}", f"user{i}@example.com")
        for i in range(count)
    ]

    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    return current, peak

---

## ▶️ Experiment 3 — Scaling the Number of Objects

A difference that looks insignificant for one object can become important when an application creates thousands of them.

We'll therefore repeat the experiment with several object counts.

In [4]:
object_counts = [1_000, 10_000, 50_000, 100_000]

results = []

for count in object_counts:
    regular_current, regular_peak = measure_memory(RegularUser, count)
    slotted_current, slotted_peak = measure_memory(SlottedUser, count)

    saving = regular_current - slotted_current
    saving_percent = (saving / regular_current) * 100

    results.append(
        (
            count,
            regular_current,
            slotted_current,
            saving,
            saving_percent,
        )
    )

---

## 📊 Results

Let's convert the measurements from bytes to megabytes and display them together.

The exact values can vary depending on the Python version, operating system, and runtime environment, so the important observation is the relative difference between the two implementations.

In [5]:
print(f"{'Objects':>10} {'Regular (MB)':>15} {'Slots (MB)':>15} {'Saved (MB)':>15} {'Saved (%)':>12}")
print("-" * 72)

for count, regular, slotted, saving, saving_percent in results:
    print(f"{count:>10,} {regular / 1024**2:>15.2f} {slotted / 1024**2:>15.2f} {saving / 1024**2:>15.2f} {saving_percent:>11.2f}%")

   Objects    Regular (MB)      Slots (MB)      Saved (MB)    Saved (%)
------------------------------------------------------------------------
     1,000            0.23            0.19            0.04       17.94%
    10,000            2.35            1.97            0.38       16.25%
    50,000           11.89            9.98            1.91       16.04%
   100,000           23.72           19.91            3.81       16.08%


---

## 🔍 What's Happening?

Regular Python objects normally store instance attributes in a per-instance dictionary called `__dict__`.

That flexibility allows attributes to be added dynamically, but the dictionary also consumes additional memory.

`__slots__` declares the permitted instance attributes in advance. For a simple slotted class like this, instances do not need the usual per-instance `__dict__`.

We can verify that directly.

In [6]:
regular_user = RegularUser(1, "Alice", "alice@example.com")
slotted_user = SlottedUser(1, "Alice", "alice@example.com")

print("RegularUser has __dict__:", hasattr(regular_user, "__dict__"))
print("SlottedUser has __dict__:", hasattr(slotted_user, "__dict__"))

RegularUser has __dict__: True
SlottedUser has __dict__: False


---

## ▶️ Experiment 4 — The Trade-Off

The memory reduction comes with a restriction.

A regular object can normally receive new attributes dynamically. A slotted object cannot receive attributes that were not declared in `__slots__`.

In [7]:
regular_user.role = "admin"
print("RegularUser role:", regular_user.role)

try:
    slotted_user.role = "admin"
except AttributeError as error:
    print("SlottedUser:", error)

RegularUser role: admin
SlottedUser: 'SlottedUser' object has no attribute 'role' and no __dict__ for setting new attributes


---

## 💡 Key Findings

- `__slots__` can reduce the per-instance memory overhead of Python objects.
- The difference becomes more noticeable when many instances of the same class are created.
- Regular instances normally provide a `__dict__`, while simple slotted instances can avoid that storage.
- The optimisation trades some of Python's dynamic attribute flexibility for a more fixed object structure.
- The exact memory savings depend on the Python implementation, version, platform, class design, and measurement method.

---

## 🧾 Conclusion

Yes — `__slots__` can reduce memory usage when many instances of a class share a fixed set of attributes.

That does not mean every Python class should use it. For ordinary applications with relatively few objects, the additional restriction may provide little practical benefit. But for memory-sensitive workloads containing large numbers of similar objects, `__slots__` can be a useful optimisation.

The important lesson is not simply that `__slots__` "saves memory" — it is that we can measure when that saving is significant enough to matter.

---